# A 3D gradient echo, out of one kernel and two `for` loops

`sc.modules.GRE3DTR` is one repetition. **There is deliberately no `GRE3D` imaging class**, and
this notebook is part of the reason: if a complete 3D acquisition is a kernel plus the ordering
you would have written anyway, then an imaging module would be a wrapper rather than a piece of
physics. The loops below are the acquisition policy, in the open, where you can change them.

What the kernel owns is the z axis, and it is the only axis that is not a 2D one:

| | |
|---|---|
| `slab_thickness_mm=None` | non-selective — what every official Pulseq 3D reference does. z carries a partition encode and nothing else |
| `slab_thickness_mm=…` | slab-selective. The rephasing the slab implies and the partition encoding are two moments on **one axis in one window**, and the kernel solves them together |

**Output:** two `.seq` files — `gre_3d_nonselective.seq` and `gre_3d_slab.seq` — and the nominal
k-space that [`02_simulate_and_reconstruct.ipynb`](02_simulate_and_reconstruct.ipynb) uses.

**This notebook varies the excitation mode**, which is a real user-facing choice. `02` varies
something else entirely: *within* the slab-selective mode, how the required z moment is realised.
Its two files are validation artifacts rather than further acquisition modes, and it says so.

In [ ]:
from pathlib import Path

import numpy as np
import pypulseq as pp

import seqcraft as sc

opts = pp.Opts(
    max_grad=20, grad_unit='mT/m',
    max_slew=120, slew_unit='T/m/s',
    B0=3.0,
    rf_dead_time=100e-6,
    rf_ringdown_time=30e-6,
    adc_dead_time=10e-6,
)

FOV_MM = (200.0, 200.0, 120.0)      # the encoded extent, x / y / z
MATRIX = (32, 32, 8)                # readout samples, lines, partitions
SLAB_MM = 140.0                     # the *excited* slab -- deliberately larger than fov_z
FLIP_DEG, BANDWIDTH_HZ_PX = 12.0, 250.0
DUMMIES = 8

NX, NY, NZ = MATRIX
SEQ_DIR = Path('seq')
SEQ_DIR.mkdir(exist_ok=True)

print(f'{NX} x {NY} x {NZ} over {FOV_MM} mm -> voxel '
      f'{FOV_MM[0] / NX:.2f} x {FOV_MM[1] / NY:.2f} x {FOV_MM[2] / NZ:.2f} mm')

## One repetition

Everything below is *reported* by the kernel rather than assumed by the caller. `Δkz` comes from
the **encoded** extent; the excited slab is a separate number and is larger here, which is the
ordinary choice for a full-volume acquisition — it keeps the profile's transition region out of
the reconstructed volume.

In [ ]:
tr = sc.modules.GRE3DTR(opts=opts, fov_mm=FOV_MM, matrix=MATRIX, slab_thickness_mm=SLAB_MM,
                        flip_deg=FLIP_DEG, bandwidth_hz_px=BANDWIDTH_HZ_PX)

print(f'winder            {tr.winder_s * 1e6:7.1f} us   (shared by x, y and z)')
print(f'limiting partition{tr.limiting_partition:5d}     of 0 ... {NZ - 1}')
print(f'TE / TR           {tr.te_s * 1e3:7.3f} / {tr.tr_s * 1e3:.3f} ms')
print(f'dk  x / y / z     ' + ' / '.join(f'{tr.dk_per_m(a):.4f}' for a in 'xyz') + '  1/m')
print(f'centre line / partition   {tr.center_line} / {tr.center_partition}')
print(f'slab rephasing    {tr.slab_rephase_area_per_m:7.2f} 1/m   '
      f'(slab {SLAB_MM:.0f} mm vs fov_z {FOV_MM[2]:.0f} mm)')

### The z axis, partition by partition

`A_z(p) = A_slab + A_partition(p)`, and **both terms are signed**. The limiting partition is
whichever one needs the longest gradient *after* that sum — not the last index, and not the
largest partition moment before combining. Reverse the slab gradient and the limit moves to the
other edge of k-space.

In [ ]:
print(f'{"p":>2}  {"A_partition":>12}  {"A_slab":>8}  {"A_z(p)":>10}')
for p in range(NZ):
    mark = '  <- limiting' if p == tr.limiting_partition else ''
    print(f'{p:2d}  {tr.pe_z.k_per_m(p):12.3f}  {tr.slab_rephase_area_per_m:8.2f}  '
          f'{tr.combined_z_area_per_m(p):10.3f}{mark}')

steps = np.diff([tr.combined_z_area_per_m(p) for p in range(NZ)])
print(f'\nadjacent partitions differ by {steps.mean():.4f} 1/m; dkz is {tr.dk_per_m("z"):.4f} '
      f'-- the slab term is constant and cancels')

## The scan: two loops and nothing else

Partition outer, line inner, with RF spoiling as the usual quadratic schedule. Dummy repetitions
come first and sample nothing. **This is the whole acquisition policy** — reorder it, subsample
it, or centre-order it by editing these four lines.

In [ ]:
def phase_deg(n, increment=117.0):
    """The standard quadratic RF-spoiling schedule, counted from the first dummy."""
    return 0.5 * increment * n * (n + 1)


def scan(kernel, *, dummies=DUMMIES):
    """One TR per (partition, line), after `dummies` that load the gradients and sample nothing."""
    table = [(line, partition) for partition in range(NZ) for line in range(NY)]
    out = sc.LogicBlock('gre_3d')
    for n in range(dummies):
        out.add(n * kernel.tr_s, kernel(line=table[0][0], partition=table[0][1],
                                        phase_deg=phase_deg(n), acquire=False))
    for index, (line, partition) in enumerate(table):
        n = dummies + index
        out.add(n * kernel.tr_s, kernel(line=line, partition=partition, phase_deg=phase_deg(n)))
    return out, table


tree, table = scan(tr)
print(f'{len(table)} repetitions + {DUMMIES} dummies = {tree.duration:.2f} s')

## What was actually encoded

Read back off the compiled sequence rather than off the design: every partition and line acquired
exactly once, on the lattice, with `kz` **held** through each readout — it is an encode, not a
rephasing.

In [ ]:
seq = sc.compile(tree, opts, name='gre_3d_slab', definitions={
    'FOV': [FOV_MM[0] / 1e3, FOV_MM[1] / 1e3, FOV_MM[2] / 1e3],
    'kSpaceCenterLine': tr.center_line,
    'kSpaceCenterPartition': tr.center_partition,
    'TE': tr.te_s, 'TR': tr.tr_s, 'SlabThickness': SLAB_MM / 1e3,
})
k = sc.kspace(tree, opts)
samples = tr.ro.num_samples
per_tr = k['k_adc'].reshape(3, len(table), samples)
echo = per_tr[:, :, tr.ro.echo_sample(0)]

ky_index = np.round(echo[1] / tr.dk_per_m('y')).astype(int)
kz_index = np.round(echo[2] / tr.dk_per_m('z')).astype(int)
wanted_y = np.array([line - tr.center_line for line, _ in table])
wanted_z = np.array([partition - tr.center_partition for _, partition in table])

print(f'{len(seq.block_events)} blocks, {seq.duration()[0]:.2f} s')
print(f'|kx| at the echo          {np.abs(echo[0]).max():.2e} 1/m')
print(f'ky index errors           {np.abs(ky_index - wanted_y).max()}')
print(f'kz index errors           {np.abs(kz_index - wanted_z).max()}')
print(f'kz spread within a TR     {np.ptp(per_tr[2], axis=1).max():.2e} 1/m')
print(f'every (ky, kz) once       {len(set(zip(ky_index, kz_index))) == len(table)}')

### TE does not depend on `kz`

One winder duration serves every partition. Letting each take its own shortest would be a
contrast gradient across the volume — invisible to every k-space check above.

In [ ]:
t_echo = k['t_adc'].reshape(len(table), samples)[:, tr.ro.echo_sample(0)]
shot_start = np.array([(DUMMIES + i) * tr.tr_s for i in range(len(table))])
te_measured = t_echo - shot_start - tr.exc.time_to_center()

print(f'TE spread over all {len(table)} repetitions: {np.ptp(te_measured) * 1e9:.3f} ns')
print(f'reported {tr.te_s * 1e3:.4f} ms, measured {te_measured.mean() * 1e3:.4f} ms')

## The non-selective path

The same kernel with `slab_thickness_mm=None`. `A_slab` is then zero, so the combined winder *is*
the partition encode — the selective implementation reduces to the reference behaviour rather
than being a separate code path.

In [ ]:
plain = sc.modules.GRE3DTR(opts=opts, fov_mm=FOV_MM, matrix=MATRIX, slab_thickness_mm=None,
                           flip_deg=FLIP_DEG, bandwidth_hz_px=BANDWIDTH_HZ_PX)

print(f'A_slab {plain.slab_rephase_area_per_m:.1f} 1/m, and the combined winder is the encode:')
print('  ', all(plain.combined_z_area_per_m(p) == plain.pe_z.k_per_m(p) for p in range(NZ)))
print(f'winder {plain.winder_s * 1e6:.1f} us vs {tr.winder_s * 1e6:.1f} us selective   '
      f'| TE {plain.te_s * 1e3:.3f} vs {tr.te_s * 1e3:.3f} ms')

plain_tree, _ = scan(plain)
plain_seq = sc.compile(plain_tree, opts, name='gre_3d_nonselective', definitions={
    'FOV': [FOV_MM[0] / 1e3, FOV_MM[1] / 1e3, FOV_MM[2] / 1e3],
    'kSpaceCenterLine': plain.center_line,
    'kSpaceCenterPartition': plain.center_partition,
    'TE': plain.te_s, 'TR': plain.tr_s,
})

## The refusals

Timing that cannot be reached is refused with the number that fixes it, and the message names
**which partition** is limiting — because with a signed slab term that is not something a caller
can work out from the matrix.

In [ ]:
try:
    sc.modules.GRE3DTR(opts=opts, fov_mm=FOV_MM, matrix=MATRIX, slab_thickness_mm=SLAB_MM,
                       bandwidth_hz_px=BANDWIDTH_HZ_PX, te_s=1e-4)
except sc.ConfigurationError as error:
    print('\n'.join(str(error).splitlines()[:7]))

try:
    sc.modules.GRE3DTR(opts=opts, fov_mm=(200.0, 200.0), matrix=MATRIX)
except sc.ConfigurationError as error:
    print('\n' + str(error).splitlines()[0])

## The files

In [ ]:
seq.write(str(SEQ_DIR / 'gre_3d_slab.seq'))
plain_seq.write(str(SEQ_DIR / 'gre_3d_nonselective.seq'))
np.savez(
    SEQ_DIR / 'gre_3d_nominal.npz',
    fov_mm=np.array(FOV_MM), matrix=np.array(MATRIX), slab_mm=SLAB_MM,
    flip_deg=FLIP_DEG, bandwidth_hz_px=BANDWIDTH_HZ_PX, dummies=DUMMIES,
    te_s=tr.te_s, tr_s=tr.tr_s, center_line=tr.center_line,
    center_partition=tr.center_partition, table=np.array(table), k_adc=per_tr,
)
print(f'-> {SEQ_DIR}/gre_3d_slab.seq, gre_3d_nonselective.seq, gre_3d_nominal.npz')

## What this notebook established

| | |
|---|---|
| a complete 3D acquisition is **one kernel and two loops** | which is why no `GRE3D` imaging class ships: it would wrap the ordering rather than own any physics |
| the limiting partition is a **result of the signed coupling** | printed per partition above, and it is not the last index |
| `Δkz` comes from the encoded extent, the slab is separate | 140 mm excited for a 120 mm encoded FOV, which is the ordinary full-volume choice |
| **TE does not depend on `kz`** | one winder for every partition, measured to nanoseconds |
| the non-selective path is the same solver at `A_slab = 0` | not a second implementation |

Next: [`02_simulate_and_reconstruct.ipynb`](02_simulate_and_reconstruct.ipynb) reconstructs the
volume and looks at it in three planes, which is where an axis swap or a reversed `kz` stops being
a number and becomes visible.